# Component tensor report

Inspect an explicitly selected native catalog of three CNOT graphs. This is a runnable report workflow, with supplied Compiler and Simulation providers. It is not a survey of all native or upstream examples.

- Statevector storage grows as $2^n$. Contraction width is the base-two logarithm of the largest intermediate in the selected contraction tree. The original executable filter includes width 25, so this report uses $w \leq 25$. It is a heuristic, not an execution or capacity guarantee.
- Each process has an eight-second deadline including process startup, graph selection, tensor construction and width planning. Up to four processes run concurrently. Timing columns measure separate stages, not quantum resource costs.
- These concrete graphs already contain primitive calls. The Compiler selects their order and Simulation builds the accepted factorized COPY/XOR tensors. Wider or symbolic examples require their own explicit providers.
- Measurements and quantum work registers are absent in this catalog. Primitive instructions observed after physical binding and the independent repetition model are printed separately.

Adapted from Qualtran v0.7.0, `dev_tools/tensor-report-card.ipynb`, frozen commit `bcb17bb25360ed68ce3bc084645db3e929799389`. Copyright 2024 Google LLC, Apache-2.0. See `docs/development/COMPONENT_TENSOR_REPORT_NOTEBOOK_AUDIT.md`.

In [ ]:
from collections import Counter

from examples.algorithms.component_tensor_report import report_component, selected_programs
from flagquantum.compiler.registers.register_binding import bind_register_program
from tools.development.process_deadlines import ProcessDeadlineExecutor

In [ ]:
programs = selected_programs()
observed_counts = {}
model_counts = {}
for repetitions, (name, program) in enumerate(programs, start=1):
    bound = bind_register_program(program, ((0,), (1,)), entry="cnot_chain")
    observed_counts[name] = dict(Counter(gate.name for gate in bound.circuit.instructions))
    model_counts[name] = {"cx": repetitions}
print("Measured primitive counts", observed_counts)
print("Model primitive estimates", model_counts)

In [ ]:
records = []
with ProcessDeadlineExecutor(timeout=8.0, max_workers=4) as executor:
    for name, program in programs:
        executor.submit(
            report_component,
            kwargs=dict(name=name, cls_name=type(program).__name__, component=program),
        )
    while executor.work_to_be_done:
        kwargs, record = executor.next_result()
        print("\r", f"{executor.work_to_be_done:5d} remaining", end="", flush=True)
        if record is None:
            records.append({"name": kwargs["name"], "cls": kwargs["cls_name"], "err": "Timeout"})
        else:
            records.append(record)

import pandas as pd
# Keep report views usable even if every task times out before producing fields.
df = pd.DataFrame(records).reindex(
    columns=["name", "cls", "flat_dur", "tn_dur", "width", "width_dur", "err"]
)

## Number of selected component examples

In [ ]:
print(len(df))

## Number of examples with completed graph selection

In [ ]:
print(len(df[df['flat_dur'] > 0]))

## Number of examples with a positive contraction width

In [ ]:
print(len(df[df['width'] > 0]))

## Examples passing the width heuristic

In [ ]:
print(len(df[df['width'] <= 25]))
df[df['width'] <= 25]

## Examples exceeding the width heuristic

In [ ]:
df[df['width'] > 25].sort_values(by='width')

## Examples without a reported width

In [ ]:
df[df['width'].isna()]

## Slowest graph selection

In [ ]:
df.sort_values(by='flat_dur', ascending=False).head()

The original notebook observed flattening as a bottleneck and tensor/width construction below half a second in its run. Those historical observations do not establish current native performance. The following tables report measured stage durations for this selected catalog only.

In [ ]:
# Slowest tn_dur
df.sort_values(by='tn_dur', ascending=False).head()

In [ ]:
# Slowest width_dur
df.sort_values(by='width_dur', ascending=False).head()